<a href="https://colab.research.google.com/github/fernandosuarez89/fernandosuarez89.github.io/blob/main/SEPA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

def distance_haversine(lat1, lon1, lat2, lon2, radius=6371):
    """
    Calculates the distance between two points on Earth using the Haversine formula.

    Args:
        lat1 (float or str): Latitude of the first point in degrees.
        lon1 (float or str): Longitude of the first point in degrees.
        lat2 (float or str): Latitude of the second point in degrees.
        lon2 (float or str): Longitude of the second point in degrees.
        radius (float, optional): Radius of the Earth in kilometers. Defaults to 6371.

    Returns:
        float or None: Distance between the two points in the same unit as the radius,
                       or None if coordinates cannot be converted to float.
    """
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
    except (ValueError, TypeError):
        return None # Return None if conversion fails for any coordinate

    # Convert latitude and longitude to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Calculate differences
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Apply Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Calculate distance
    distance = radius * c
    return distance

# Example usage:
lat1, lon1 = 40.7128, -74.0060  # New York
lat2, lon2 = 34.0522, -118.2437  # Los Angeles
distance = distance_haversine(lat1, lon1, lat2, lon2)
print(f"The distance between New York and Los Angeles is: {distance:.2f} km")

The distance between New York and Los Angeles is: 3935.75 km


In [ ]:
import requests
import zipfile
import os
import pandas as pd
from datetime import datetime, timedelta
import pytz


def clean_column_names(df, prefixes):
    new_columns = []
    for col in df.columns:
        cleaned_col = col
        for prefix in prefixes:
            if cleaned_col.startswith(prefix):
                cleaned_col = cleaned_col[len(prefix):]
                break
        new_columns.append(cleaned_col)
    df.columns = new_columns
    return df


sepas = {
    "lunes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/0a9069a9-06e8-4f98-874d-da5578693290/download/sepa_lunes.zip",
    "martes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/9dc06241-cc83-44f4-8e25-c9b1636b8bc8/download/sepa_martes.zip",
    "miercoles": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/1e92cd42-4f94-4071-a165-62c4cb2ce23c/download/sepa_miercoles.zip",
    "jueves": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/d076720f-a7f0-4af8-b1d6-1b99d5a90c14/download/sepa_jueves.zip",
    "viernes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/91bc072a-4726-44a1-85ec-4a8467aad27e/download/sepa_viernes.zip",
    "sabado": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/b3c3da5d-213d-41e7-8d74-f23fda0a3c30/download/sepa_sabado.zip",
    "domingo": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/f8e75128-515a-436e-bf8d-5c63a62f2005/download/sepa_domingo.zip"
}


# Define Buenos Aires timezone
buenos_aires_tz = pytz.timezone('America/Argentina/Buenos_Aires')

# Get current time in Buenos Aires
now_ba = datetime.now(buenos_aires_tz)

# Determine the date for the data to fetch
if now_ba.hour >= 15:  # 3 PM (15:00) cutoff
    target_date = now_ba.date()
elif now_ba.hour < 15:
    target_date = now_ba.date() - timedelta(days=1)

# Get the day of the week (0=Monday, 6=Sunday)
day_of_week_num = target_date.weekday()

day_names = [
    "lunes",
    "martes",
    "miercoles",
    "jueves",
    "viernes",
    "sabado",
    "domingo",
]

# Get the corresponding day name
target_day_name = day_names[day_of_week_num]

# Initialize a flag to control processing
should_process_zip = False

# Ensure zip_file_name is always set
zip_file_name = f"sepa_{target_day_name}.zip"

# First attempt to get the URL
current_zip_url = sepas.get(target_day_name)

if not current_zip_url:
    print(f"Error: No ZIP URL found for {target_day_name} in the URL dictionary.")
    if os.path.exists(zip_file_name):
        print(f"However, ZIP file '{zip_file_name}' found locally. Attempting to use local file.")
        should_process_zip = True # Trust existing file for now, will validate before extraction
    else:
        print(f"And '{zip_file_name}' not found locally. Skipping file extraction and processing.")
else: # current_zip_url is valid
    if os.path.exists(zip_file_name):
        print(f"ZIP file '{zip_file_name}' already exists locally. Checking validity...")
        try:
            # Attempt to open as a zip file to validate it
            with zipfile.ZipFile(zip_file_name, 'r') as zip_test:
                zip_test.testzip() # This method checks the integrity of the archive
            print(f"Local ZIP file '{zip_file_name}' is valid. Skipping download.")
            should_process_zip = True
        except zipfile.BadZipFile:
            print(f"Warning: Local file '{zip_file_name}' is corrupt or not a valid ZIP file. Deleting and attempting re-download.")
            os.remove(zip_file_name) # Delete corrupt file
            # Fall through to download logic below
        except Exception as e:
            print(f"Error validating local ZIP file '{zip_file_name}': {e}. Deleting and attempting re-download.")
            os.remove(zip_file_name)
            # Fall through to download logic below

    # If no valid local file was found (either didn't exist, or was corrupt and deleted), then download
    if not should_process_zip:
        # Add headers to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        try:
            print(f"Attempting to download '{zip_file_name}' from {current_zip_url}.")
            response = requests.get(current_zip_url, headers=headers)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

            with open(zip_file_name, 'wb') as f:
                f.write(response.content)

            print(f"ZIP file '{zip_file_name}' downloaded successfully.")
            should_process_zip = True # Set to true if download was successful
        except requests.exceptions.RequestException as e:
            print(f"Error downloading ZIP file: {e}")
            print(f"Please try to manually download '{zip_file_name}' from the website and upload it to your Colab environment.")
            should_process_zip = False # Download failed, do not process

# Now, the main processing block, proceed only if should_process_zip is True and the file exists
if should_process_zip and os.path.exists(zip_file_name):
    # Final verification of the ZIP file before extraction to catch issues with newly downloaded files
    try:
        with zipfile.ZipFile(zip_file_name, 'r') as zip_test_extract:
            zip_test_extract.testzip() # This also raises BadZipFile if corrupt
    except zipfile.BadZipFile:
        print(f"Critical Error: '{zip_file_name}' is not a valid ZIP file even after (re)download/local check. Skipping extraction and further processing.")
        if os.path.exists(zip_file_name):
            os.remove(zip_file_name)
            print(f"Removed corrupt file '{zip_file_name}'.")
        should_process_zip = False
    except Exception as e:
        print(f"Critical Error: An unexpected error occurred while verifying '{zip_file_name}': {e}. Skipping extraction and further processing.")
        if os.path.exists(zip_file_name):
            os.remove(zip_file_name)
            print(f"Removed problematic file '{zip_file_name}'.")
        should_process_zip = False

if should_process_zip:
    # Directory to extract the contents to
    extract_dir = target_date.strftime('%Y-%m-%d')
    os.makedirs(extract_dir, exist_ok=True)

    # Create a ZipFile object and extract all contents
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Contents of '{zip_file_name}' extracted to '{os.path.abspath(extract_dir)}' successfully.")

    # Determine the actual directory containing the files to process
    # This handles cases where the main ZIP extracts into a single top-level folder
    actual_content_dir = extract_dir
    extracted_items = os.listdir(extract_dir)
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(extract_dir, extracted_items[0])):
        actual_content_dir = os.path.join(extract_dir, extracted_items[0])

    # Now, nested_zip_search_dir should point to the directory that actually contains the zips to be extracted
    nested_zip_search_dir = actual_content_dir

    # List all files in the nested_zip_search_dir
    all_files_in_nested_dir = os.listdir(nested_zip_search_dir)

    # Filter for .zip files (these are the inner zips)
    nested_zip_files = [f for f in all_files_in_nested_dir if f.endswith('.zip')]

    print(f"Found {len(nested_zip_files)} nested ZIP files in '{nested_zip_search_dir}':")
    for nested_zip_file_name in nested_zip_files:
        nested_zip_file_path = os.path.join(nested_zip_search_dir, nested_zip_file_name)

        # Create a sub-directory for each nested zip's extraction to avoid clutter
        # Extract into a directory within actual_content_dir
        extraction_sub_dir = os.path.join(actual_content_dir, nested_zip_file_name.replace('.zip', ''))
        os.makedirs(extraction_sub_dir, exist_ok=True)

        try:
            with zipfile.ZipFile(nested_zip_file_path, 'r') as nested_zip_ref:
                nested_zip_ref.extractall(extraction_sub_dir)
            print(f"  - Extracted '{nested_zip_file_name}' to '{extraction_sub_dir}'")
        except zipfile.BadZipFile:
            print(f"  - Warning: '{nested_zip_file_name}' is not a valid ZIP file and could not be extracted.")
            if os.path.exists(nested_zip_file_path):
                os.remove(nested_zip_file_path)
                print(f"    Removed corrupt nested file '{nested_zip_file_name}'.")
        except Exception as e:
            print(f"  - Error extracting '{nested_zip_file_name}': {e}")

    print("Finished extracting all identified nested ZIP files.")

    csv_files_found = []
    search_dir = actual_content_dir # Scan for CSVs starting from the actual content directory

    # Walk through the directory and its subdirectories
    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_files_found.append(os.path.join(root, file))

    print(f"Found {len(csv_files_found)} CSV files:")


    # Initialize lists to store categorized files and their sources
    productos_files = []
    sucursales_files = []
    comercio_files = []

    # Iterate through each file path in the csv_files_found list
    for file_path in csv_files_found:
        # Extract the base name of the file (e.g., 'productos.csv')
        file_name = os.path.basename(file_path)
        # Extract the directory path of the file (the parent directory)
        file_source = os.path.dirname(file_path)

        # Categorize the file based on its name and append to the corresponding list
        if 'productos.csv' in file_name:
            productos_files.append((file_path, file_source))
        elif 'sucursales.csv' in file_name:
            sucursales_files.append((file_path, file_source))
        elif 'comercio.csv' in file_name:
            comercio_files.append((file_path, file_source))
        else:
            print(f"Warning: Uncategorized CSV file found: {file_path}")

    print(f"Total productos files found: {len(productos_files)}")
    print(f"Total sucursales files found: {len(sucursales_files)}")
    print(f"Total comercio files found: {len(comercio_files)}")
else:
    print("Skipping file extraction and processing due to unresolvable issues (download failed or no valid local file found).")

ZIP file 'sepa_viernes.zip' already exists locally. Checking validity...
Local ZIP file 'sepa_viernes.zip' is valid. Skipping download.
Contents of 'sepa_viernes.zip' extracted to '/content/2026-10-23' successfully.
Found 17 nested ZIP files in '2026-10-23/2026-10-23':
  - Extracted 'sepa_2_comercio-sepa-6_2026-10-23_01-05-07.zip' to '2026-10-23/2026-10-23/sepa_2_comercio-sepa-6_2026-10-23_01-05-07'
  - Extracted 'sepa_1_comercio-sepa-4_2026-10-23_09-05-11.zip' to '2026-10-23/2026-10-23/sepa_1_comercio-sepa-4_2026-10-23_09-05-11'
  - Extracted 'sepa_2_comercio-sepa-11_2026-10-23_01-05-07.zip' to '2026-10-23/2026-10-23/sepa_2_comercio-sepa-11_2026-10-23_01-05-07'
  - Extracted 'sepa_1_comercio-sepa-21_2026-10-23_09-05-11.zip' to '2026-10-23/2026-10-23/sepa_1_comercio-sepa-21_2026-10-23_09-05-11'
  - Warning: 'sepa_2_comercio-sepa-36_2026-10-23_01-05-07.zip' is not a valid ZIP file and could not be extracted.
    Removed corrupt nested file 'sepa_2_comercio-sepa-36_2026-10-23_01-05-07.zi

In [ ]:
import pandas as pd
import numpy as np

def robust_load_csv_simplified(file_path, sep, dtype, target_cols=None):
    """
    Loads a CSV file simply, cleans column names, and extracts only the available target columns
    to avoid memory crashes while tolerating files with missing or differently-named columns.
    """
    try:
        # Load the CSV file. We don't use 'usecols' inside read_csv to avoid strict key errors.
        df = pd.read_csv(file_path, sep=sep, dtype=dtype, low_memory=False)
        if df.empty:
            return df

        # Clean column headers (strip leading/trailing whitespaces)
        df.columns = df.columns.str.strip()

        # Clean footer metadata rows
        first_col = df.columns[0]
        mask_footer = df[first_col].astype(str).str.contains('Última|actualización|actualizacion', case=False, na=True)
        df = df[~mask_footer]
        df = df.dropna(how='all')

        # Filter only the columns that actually exist in this file
        if target_cols:
            existing_target_cols = [col for col in target_cols if col in df.columns]
            df = df[existing_target_cols]

        return df
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()

# Target columns matching the user's requirements (both with and without prefix for compatibility)
productos_cols = [
    'id_comercio', 'id_bandera', 'id_sucursal', 'id_producto',
    'productos_descripcion', 'productos_precio_lista',
    'productos_precio_unitario_promo1', 'productos_leyenda_promo1',
    'productos_precio_unitario_promo2', 'productos_leyenda_promo2',
    'descripcion', 'precio_lista', 'precio_unitario_promo1',
    'leyenda_promo1', 'precio_unitario_promo2', 'leyenda_promo2'
]

all_productos_dfs = []
for file_path, file_source in productos_files:
    df = robust_load_csv_simplified(
        file_path,
        sep='|',
        dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str, 'id_producto': float},
        target_cols=productos_cols
    )
    if not df.empty:
        df['file_source'] = file_source
        all_productos_dfs.append(df)

if all_productos_dfs:
    all_productos_df = pd.concat(all_productos_dfs, ignore_index=True)
    productos_prefixes = ['productos_']
    all_productos_df.columns = all_productos_df.columns.str.strip()
    all_productos_df = clean_column_names(all_productos_df, productos_prefixes)
    print(f"Loaded products successfully. Columns present: {list(all_productos_df.columns)}")
    print(f"Current memory usage: {all_productos_df.memory_usage().sum() / 1e6:.2f} MB")
else:
    print("No productos files were loaded.")

# Loading sucursales and comercio
all_sucursales_dfs = []
for file_path, file_source in sucursales_files:
    df = robust_load_csv_simplified(
        file_path,
        sep='|',
        dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}
    )
    if not df.empty:
        df['file_source'] = file_source
        all_sucursales_dfs.append(df)

if all_sucursales_dfs:
    all_sucursales_df = pd.concat(all_sucursales_dfs, ignore_index=True)
    sucursales_prefixes = ['sucursales_']
    all_sucursales_df.columns = all_sucursales_df.columns.str.strip()
    all_sucursales_df = clean_column_names(all_sucursales_df, sucursales_prefixes)
else:
    print("No sucursales files were loaded.")

all_comercio_dfs = []
for file_path, file_source in comercio_files:
    df = robust_load_csv_simplified(
        file_path,
        sep='|',
        dtype={'id_comercio': str, 'id_bandera': str}
    )
    if not df.empty:
        df['file_source'] = file_source
        all_comercio_dfs.append(df)

if all_comercio_dfs:
    all_comercio_df = pd.concat(all_comercio_dfs, ignore_index=True)
    comercio_prefixes = ['comercio_']
    all_comercio_df.columns = all_comercio_df.columns.str.strip()
    all_comercio_df = clean_column_names(all_comercio_df, comercio_prefixes)
else:
    print("No comercio files were loaded.")

Loaded products successfully. Columns present: ['id_comercio', 'id_bandera', 'id_sucursal', 'id_producto', 'descripcion', 'precio_lista', 'precio_unitario_promo1', 'leyenda_promo1', 'precio_unitario_promo2', 'leyenda_promo2', 'file_source']
Current memory usage: 1249.73 MB


In [ ]:
bandera_filtro = ['Changomas', 'COTO CICSA', 'Disco', 'Express', 'HiperChangomas', 'Hipermercado Carrefour', 'Jumbo', 'Market', 'Maxi', 'SuperChangomas', 'Supermercados DIA','FARMACITY','SIMPLICITY']
my_location = ['-34.53535860586283', '-58.47559604248442']
sucursales = all_sucursales_df.drop(columns = [ 'lunes_horario_atencion', 'martes_horario_atencion', 'miercoles_horario_atencion', 'jueves_horario_atencion', 'viernes_horario_atencion', 'sabado_horario_atencion', 'domingo_horario_atencion'])
sucursales['distance'] = sucursales.apply(lambda row: distance_haversine(my_location[0], my_location[1], row['latitud'], row['longitud']), axis=1)
# all_sucursales_df[all_sucursales_df['bandera'].isin(bandera_filtro)]
sucursales_2 = sucursales[(sucursales['distance'] <= 6) |
                          (sucursales['distance'].isna())]
# sucursales_3 = sucursales[(sucursales['id_comercio'] == "9") & (sucursales['id_sucursal'] == "5202")]
# sucursales_2

In [ ]:
# Define the list of specific product IDs to filter
headers = ["id_producto", "marca_ok", "segmento", "talle", "unidades"]

data = [
    [7500435228756, "Pampers", "Babydry", "P", 56],
    [7500435247368, "Pampers", "Babydry", "P", 36],
    [7500435237611, "Pampers", "Deluxe", "P", 36],
    # [, "Pampers", "Deluxe", "P", 56],
    [7500435228633, "Pampers", "Babysan", "P", 12],
    [7500435230506, "Pampers", "Babysan", "P", 52],
    [7790250042181, "Babysec", "Super Premium", "M", 48],
    # [7790250042211, "Babysec", "Super Premium", "P", 48],
    [7790250042792, "Babysec", "Premium", "M", 48],
    [7790250047230, "Babysec", "Premium", "M", 68],
    [7794626012969, "Huggies", "Flexi Confort", "M", 22], #no lo encuentra
    [7794626012839, "Huggies", "Flexi Confort", "M", 48],
    [7794626012853, "Huggies", "Flexi Confort", "M", 68],
    [7794626010019, "Huggies", "Supreme Care", "M", 22],
    [7794626013270, "Huggies", "Supreme Care", "M", 48],
    [7794626013317, "Huggies", "Supreme Care", "M", 68]
]

# Create a DataFrame from the product data
product_details_df = pd.DataFrame(data, columns=headers)

# Extract the specific_id_productos from the new DataFrame
specific_id_productos = product_details_df['id_producto'].tolist()

# Filter all_productos_df for these specific product IDs
filtered_productos_df_temp = all_productos_df[all_productos_df['id_producto'].isin(specific_id_productos)].copy()

# Merge with product_details_df to add marca_ok, segmento, talle, unidades
filtered_productos_df = pd.merge(
    filtered_productos_df_temp,
    product_details_df,
    on='id_producto',
    how='left'
)

print(f"Filtered {len(filtered_productos_df)} product entries with the specified IDs.")
# display(filtered_productos_df.head())

Filtered 4776 product entries with the specified IDs.


In [ ]:
merged_with_filtered_sucursales = pd.merge(
    filtered_productos_df,
    sucursales_2,
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='inner' # Use inner merge to only keep matching products and stores
)

final_merged_filtered_data = pd.merge(
    merged_with_filtered_sucursales,
    all_comercio_df,
    on=['id_comercio', 'id_bandera'],
    how='inner' # Use inner merge to only keep matching products, stores, and commercial entities
)

print("Merged filtered products with specific sucursales and comercio (id_comercio = '10').")
# display(final_merged_filtered_data.head())

Merged filtered products with specific sucursales and comercio (id_comercio = '10').


In [ ]:
import numpy as np

selected_columns = [
    'id_producto',
    'descripcion',
    'cantidad_presentacion',
    'unidad_medida_presentacion',
    'marca',
    'precio_lista',
    'precio_referencia',
    'cantidad_referencia',
    'unidad_medida_referencia',
    'precio_unitario_promo1',
    'leyenda_promo1',
    'precio_unitario_promo2',
    'leyenda_promo2',
    'file_source_x',
    'nombre',
    'calle',
    'numero',
    'bandera_nombre',
    'marca_ok',
    'segmento',
    'talle',
    'unidades'
]

# Dynamically filter selected_columns to only keep those that actually exist in final_merged_filtered_data
existing_columns = [col for col in selected_columns if col in final_merged_filtered_data.columns]

# Filter the DataFrame to keep only the existing selected columns
final_merged_filtered_data_selected = final_merged_filtered_data[existing_columns].copy()

# Ensure 'unidades' is numeric (important for division) and handle potential NaN values if it exists
if 'unidades' in final_merged_filtered_data_selected.columns:
    final_merged_filtered_data_selected['unidades'] = pd.to_numeric(final_merged_filtered_data_selected['unidades'], errors='coerce')

    # Calculate potential unit prices by dividing by 'unidades'
    promo2_divided = final_merged_filtered_data_selected.get('precio_unitario_promo2', np.nan) / final_merged_filtered_data_selected['unidades']
    promo1_divided = final_merged_filtered_data_selected.get('precio_unitario_promo1', np.nan) / final_merged_filtered_data_selected['unidades']
    lista_divided = final_merged_filtered_data_selected.get('precio_lista', np.nan) / final_merged_filtered_data_selected['unidades']

    # Replace infinite values (e.g., from division by zero) with NaN so fillna can work correctly
    promo2_divided = promo2_divided.replace([np.inf, -np.inf], np.nan) if hasattr(promo2_divided, 'replace') else promo2_divided
    promo1_divided = promo1_divided.replace([np.inf, -np.inf], np.nan) if hasattr(promo1_divided, 'replace') else promo1_divided
    lista_divided = lista_divided.replace([np.inf, -np.inf], np.nan) if hasattr(lista_divided, 'replace') else lista_divided

    # Apply coalesce logic: use promo2_divided if available, else promo1_divided, else lista_divided
    final_merged_filtered_data_selected['precio_unitario_por_unidad'] = (
        promo2_divided
        .fillna(promo1_divided)
        .fillna(lista_divided)
    )

    # Sort the DataFrame by the new calculated unit price
    final_merged_filtered_data_selected = final_merged_filtered_data_selected.sort_values(by='precio_unitario_por_unidad', ascending=True)

print("DataFrame with available selected columns, calculated unit price, and sorted:")
display(final_merged_filtered_data_selected)

DataFrame with available selected columns, calculated unit price, and sorted:


,id_producto,descripcion,precio_lista,precio_unitario_promo1,leyenda_promo1,precio_unitario_promo2,leyenda_promo2,file_source_x,nombre,calle,numero,bandera_nombre,marca_ok,segmento,talle,unidades,precio_unitario_por_unidad
273,7.500435e+12,PAÑAL BABYDRY HIP P,19770.0,NaN,NaN,11862.0,40% de descuento. 40% PAMPERS BABYDRY - Vigenc...,2026-10-23/2026-10-23/sepa_1_comercio-sepa-15_...,5134 - MUNRO,AV BARTOLOME MITRE,2671.0,Supermercados DIA,Pampers,Babydry,P,36,329.500000
269,7.500435e+12,PAÑAL BABYDRY HIP P,19770.0,NaN,NaN,11862.0,40% de descuento. 40% PAMPERS BABYDRY - Vigenc...,2026-10-23/2026-10-23/sepa_1_comercio-sepa-15_...,1097 - CAPITAL FEDERAL,CRAMER,2060.0,Supermercados DIA,Pampers,Babydry,P,36,329.500000
271,7.500435e+12,PAÑAL BABYDRY HIP P,19770.0,NaN,NaN,11862.0,40% de descuento. 40% PAMPERS BABYDRY - Vigenc...,2026-10-23/2026-10-23/sepa_1_comercio-sepa-15_...,1524 - PARQUE CHAS,AV DE LOS CONSTITUYENTES,3836.0,Supermercados DIA,Pampers,Babydry,P,36,329.500000
113,7.500435e+12,PAÑAL BABYDRY HIP P,19770.0,NaN,NaN,11862.0,40% de descuento. 40% PAMPERS BABYDRY - Vigenc...,2026-10-23/2026-10-23/sepa_1_comercio-sepa-15_...,84 - FLORIDA,CL SAN MARTIN,2269.0,Supermercados DIA,Pampers,Babydry,P,36,329.500000
117,7.500435e+12,PAÑAL BABYDRY HIP P,19770.0,NaN,NaN,11862.0,40% de descuento. 40% PAMPERS BABYDRY - Vigenc...,2026-10-23/2026-10-23/sepa_1_comercio-sepa-15_...,108 - VILLA URQUIZA,AV MONROE,4278.0,Supermercados DIA,Pampers,Babydry,P,36,329.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,7.794626e+12,PANALES M AHORROPACK NATURAL CARE HUGGIES X 68...,56035.0,NaN,NaN,NaN,NaN,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Villa Urquiza,Av. Roosevelt,5749.0,Hipermercado Carrefour,Huggies,Supreme Care,M,68,824.044118
319,7.794626e+12,PANALES M AHORROPACK NATURAL CARE HUGGIES X 68...,56035.0,NaN,NaN,NaN,NaN,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Ugarte,Gob. Ugarte,1980.0,Hipermercado Carrefour,Huggies,Supreme Care,M,68,824.044118
296,7.794626e+12,PANALES M AHORROPACK NATURAL CARE HUGGIES X 68...,56035.0,NaN,NaN,NaN,NaN,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Vicente López,Av. del Libertador,215.0,Hipermercado Carrefour,Huggies,Supreme Care,M,68,824.044118
288,7.790250e+12,BABYSEC SUPERPREMIUM,40249.0,NaN,NaN,NaN,NaN,2026-10-23/2026-10-23/sepa_2_comercio-sepa-11_...,Grafa,AV. DE LOS CONSTITUYENTES,6020.0,HiperChangomas,Babysec,Super Premium,M,48,838.520833


In [ ]:
# Save the final merged DataFrame to a CSV file
output_csv_filename = 'filtered_products_with_nearby_stores.csv'
final_merged_filtered_data_selected.to_csv(output_csv_filename, index=False)

print(f"The merged data has been saved to '{output_csv_filename}'.")

The merged data has been saved to 'filtered_products_with_nearby_stores.csv'.


In [ ]:
import sys
import pandas as pd

# Get all pandas DataFrames in the local and global scope
local_vars = list(globals().items()) + list(locals().items())
dataframe_info = []

for name, val in local_vars:
    if isinstance(val, pd.DataFrame):
        # Calculate memory in Megabytes
        mem_usage_mb = val.memory_usage(deep=True).sum() / (1024 ** 2)
        dataframe_info.append({
            'Variable Name': name,
            'Rows': val.shape[0],
            'Columns': val.shape[1],
            'Memory Usage (MB)': round(mem_usage_mb, 2)
        })

# Create a summary DataFrame
if dataframe_info:
    df_ram_usage = pd.DataFrame(dataframe_info).drop_duplicates(subset=['Variable Name'])
    df_ram_usage = df_ram_usage.sort_values(by='Memory Usage (MB)', ascending=False).reset_index(drop=True)

    # Display the final RAM consumption table
    print("=== RAM Consumption by Active DataFrames ===")
    display(df_ram_usage)
else:
    print("No active pandas DataFrames found in the current workspace memory.")

=== RAM Consumption by Active DataFrames ===


,Variable Name,Rows,Columns,Memory Usage (MB)
0,all_productos_df,14201421,11,6162.98
1,all_sucursales_df,2771,22,3.20
2,filtered_productos_df,4776,15,3.04
3,filtered_productos_df_temp,4776,11,2.30
4,sucursales,2771,16,2.09
5,final_merged_filtered_data,321,35,0.54
6,merged_with_filtered_sucursales,321,28,0.41
7,sucursales_2,341,16,0.26
8,final_merged_filtered_data_selected,321,17,0.25
9,all_comercio_df,28,9,0.01


### Inferring Product Attributes from Grouped Descriptions

This section filters products by common brands (Huggies, Pampers, Babysec), aggregates all unique descriptions for each `id_producto`, and then attempts to infer `segmento` (product line), `talle` (size), and `unidades` (quantity) using regular expressions.

In [ ]:
import re

# 1. Filter all_productos_df for relevant brands
brand_keywords = ['huggies', 'pampers', 'babysec','babydry']

filtered_by_brand_df = all_productos_df[
    (all_productos_df['descripcion'].fillna('').str.contains('|'.join(brand_keywords), case=False, na=False))
    & (~all_productos_df['id_comercio'].isin([2, 13]))
].copy()

# 2. Group by id_producto and aggregate descriptions
# Concatenate unique descriptions for each product
grouped_descriptions_df = filtered_by_brand_df.groupby('id_producto')['descripcion'].apply(lambda x: ' '.join(x.dropna().unique())).reset_index()
grouped_descriptions_df.rename(columns={'descripcion': 'all_descriptions'}, inplace=True)

# 3. Infer 'marca', 'segmento', 'talle', and 'unidades' from grouped descriptions

# Define brand-specific segment keywords
segment_keywords_by_brand = {
    'Pampers': {
        'Babydry': ['babydry', 'babydri'],
        'Babysan': ['babysan'],
        'Deluxe': ['deluxe'],
        'Active Sec': ['active sec'],
        'Confort Sec': ['confort sec'],
        'Premium Care': ['premium care', 'premium_care'],
        'Recien Nacido': ['recien nacido', 'newborn', 'rn']
    },
    'Babysec': {
        'Super Premium': ['super premium', 'superpremiun', 'super_premium'],
        'Premium': ['premium'],
        'Ultrasoft': ['ultrasoft']
    },
    'Huggies': {
        'Flexi Confort': ['flexi confort', 'flexi_confort'],
        'Supreme Care': ['supreme care', 'supreme_care','dermacare'],
        'Natural Care': ['natural care', 'natural_care'],
        'Protect Plus': ['protect plus'],
        'Classic': ['classic'],
        'Little Swimmers': ['little swimm', 'little swimmers']
    }
}

# Define all size keywords (from previous cells, plus some more variations)
all_size_keywords_extended = {
    'P': ['P', 'PEQUEÑO', 'PEQ', 'SMALL'],
    'M': ['M', 'MEDIANO', 'MED'],
    'XXXG': ['XXXG', 'XXX-GRANDE'],
    'XXG': ['XXG', 'XX-GRANDE'],
    'XG': ['XG', 'EXTRA GRANDE', 'X-GRANDE', 'XGD'],
    'G': ['G', 'GRANDE', 'GDE'],
    'RN': ['RN', 'RECIEN NACIDO', 'NEWBORN','R.NACIDO']
}

def infer_marca(text):
    text_upper = text.upper()
    for brand_name in brand_keywords:
        if re.search(r'\b' + re.escape(brand_name.upper()) + r'\b', text_upper):
            return brand_name.capitalize()
    return None

def infer_segmento(text, brand):
    text_upper = text.upper()
    if brand and brand in segment_keywords_by_brand:
        for segment, keywords in segment_keywords_by_brand[brand].items():
            for kw in keywords:
                if re.search(r'\b' + re.escape(kw.upper()) + r'\b', text_upper):
                    return segment
    # If brand is found but no specific segment, assign 'Others'
    if brand:
        return 'Others'
    return None

def infer_talle(text):
    text_upper = text.upper()
    # Prioritize specific sizes over general terms if both are present
    for talle, keywords in all_size_keywords_extended.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw.upper()) + r'\b', text_upper):
                return talle
    return None

def infer_unidades(text):
    text_upper = text.upper()

    # Prioritized patterns for unit extraction
    patterns = [
        r'\bX\s*(\d+)(?:\s*UNI|U)?\b',                                     # X 36, X36UNI, X36U
        r'\b(?:' + '|'.join(re.escape(k) for k in all_size_keywords_extended.keys()) + r')(\d+)\b', # MX34, XXG20
        r'\b(\d+)\s*PC\b',                                                  # 34PC
        r'(\d+)(?:\s*U|\s*UNI|\s*UNIDADES|\s*PAQ|\s*UN)\b',               # 48U, 48UNI, 48UN, 48 UNIDADES (with or without space)
        r'(\d+)\s*UN\b',                                                    # 48 UN (standalone)
        r'PAQ\s*(\d+)',                                                      # PAQ 48
        r'\b(\d+)\b$'
    ]

    for pattern in patterns:
        match = re.search(pattern, text_upper)
        if match:
            return int(match.group(1))

    return None

grouped_descriptions_df['inferred_marca'] = grouped_descriptions_df['all_descriptions'].apply(infer_marca)

# Filter out products with 'TOA' in their descriptions (as requested previously)
grouped_descriptions_df = grouped_descriptions_df[~grouped_descriptions_df['all_descriptions'].str.contains(r'\bTOA\b', case=False, na=False)].copy()

# Filter out non-diaper products explicitly (e.g., wet wipes, liquid soap, shampoo, creams, towels)
non_diaper_keywords_regex = [
    r'TOALL(?:ITAS)?(?:\s*|\.|-)?HUMED[A]?S?',
    r'JABON LIQUIDO',
    r'SHAMPOO',
    r'COLONIA',
    r'JABON\b',
    r'CREMA',
    r'TOALLAS',
    r'APOSITO'
]
non_diaper_pattern = '|'.join(non_diaper_keywords_regex)
grouped_descriptions_df = grouped_descriptions_df[~grouped_descriptions_df['all_descriptions'].str.contains(non_diaper_pattern, flags=re.IGNORECASE, na=False)].copy()

grouped_descriptions_df['inferred_segmento'] = grouped_descriptions_df.apply(lambda row: infer_segmento(row['all_descriptions'], row['inferred_marca']), axis=1)
grouped_descriptions_df['inferred_talle'] = grouped_descriptions_df['all_descriptions'].apply(infer_talle)
grouped_descriptions_df['inferred_unidades'] = grouped_descriptions_df['all_descriptions'].apply(infer_unidades)

# Display the resulting DataFrame
print("Products with grouped descriptions and inferred attributes:")
display(grouped_descriptions_df.head())


Products with grouped descriptions and inferred attributes:


,id_producto,all_descriptions,inferred_marca,inferred_segmento,inferred_talle,inferred_unidades
1,7.441008e+12,"HUGGIES DERMACARE PAÑAL P DERMACARE AP, HUGGIE...",Huggies,Supreme Care,P,NaN
2,7.500435e+12,PANAL PAMPERS REC.NACIDO RN+ 20 u.,Pampers,Recien Nacido,RN,20.0
5,7.500435e+12,PAÑAL PREMIUM CARE PANTS TALLE: M PAMPERS PAQ ...,Pampers,Premium Care,M,34.0
6,7.500435e+12,PAÑAL PREMIUM CARE PANTS TALLE: G PAMPERS PAQ ...,Pampers,Premium Care,G,30.0
7,7.500435e+12,PAÑAL PREMIUM CARE PANTS TALLE: XG PAMPERS PAQ...,Pampers,Premium Care,XG,26.0


In [ ]:
stores_and_brands#[stores_and_brands['id_comercio'] == 3]

,id_comercio,id_bandera,id_sucursal,nombre,tipo,calle,numero,latitud,longitud,observaciones,barrio,codigo_postal,localidad,provincia,file_source,bandera_nombre
0,3,1,2,ACASSUSO,Autoservicio,AV. LIBERTADOR,15301.0,-34.47436,-58.50185,NaN,Acassuso,1641,ACASSUSO,AR-B,2026-10-23/2026-10-23/sepa_1_comercio-sepa-3_2...,DEHEZA S.A.I.C.F. e I.
1,3,1,3,RIVER,Autoservicio,AV. F.ALCORTA,7786.0,-34.546397,-58.451884,NaN,Belgrano,1428,CAPITAL FEDERAL,AR-C,2026-10-23/2026-10-23/sepa_1_comercio-sepa-3_2...,DEHEZA S.A.I.C.F. e I.
2,3,1,29,CABILDO,Autoservicio,CABILDO,4701.0,-34.541473,-58.47384,NaN,Nuñez,1429,CAPITAL FEDERAL,AR-C,2026-10-23/2026-10-23/sepa_1_comercio-sepa-3_2...,DEHEZA S.A.I.C.F. e I.
3,3,1,31,CENTENARIO,Autoservicio,AV. F. ALCORTA,3099.0,-34.575574,-58.397682,NaN,Palermo,1425,CAPITAL FEDERAL,AR-C,2026-10-23/2026-10-23/sepa_1_comercio-sepa-3_2...,DEHEZA S.A.I.C.F. e I.
4,3,1,36,LINCOLN Y NUEVA YORK,Autoservicio,AV. LINCOLN,3999.0,-34.599622,-58.5103,NaN,Villa Devoto,1419,CAPITAL FEDERAL,AR-C,2026-10-23/2026-10-23/sepa_1_comercio-sepa-3_2...,DEHEZA S.A.I.C.F. e I.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2680,10,2,258,Villa Pueyrredon,Supermercado,Av. Mosconi,2861.0,-34.585933,-58.501809,NaN,NaN,1419.0,Ciudad Autónoma de Buenos Aires,AR-C,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Market
2681,10,3,344,M. T. de Alvear 2251,Autoservicio,Marcelo T. de Alvear,2251.0,-34.596862,-58.399571,NaN,Recoleta,1122.0,Ciudad Autónoma de Buenos Aires,AR-C,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Express
2682,10,3,593,Entre Ríos 1192,Autoservicio,Entre Ríos,1192.0,-34.622603,-58.391341,NaN,San Cristobal,1080.0,Ciudad Autónoma de Buenos Aires,AR-C,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Express
2683,10,3,791,BEIRO 3342,Autoservicio,BEIRO,3345.0,-34.598084,-58.500119,NaN,NaN,1419.0,Ciudad Autónoma de Buenos Aires,AR-C,2026-10-23/2026-10-23/sepa_2_comercio-sepa-10_...,Express


In [ ]:
import numpy as np
import pandas as pd

# 1. Clean the store details and merge them with their commercial names
sucursales_full = all_sucursales_df.drop(columns=[
    'lunes_horario_atencion', 'martes_horario_atencion', 'miercoles_horario_atencion',
    'jueves_horario_atencion', 'viernes_horario_atencion', 'sabado_horario_atencion',
    'domingo_horario_atencion'
], errors='ignore')

stores_and_brands = pd.merge(
    sucursales_full[~sucursales_full['id_comercio'].isin([2,13])],
    all_comercio_df[['id_comercio', 'id_bandera', 'bandera_nombre']],
    on=['id_comercio', 'id_bandera'],
    how='inner'
)

# 2. Extract pricing for relevant brand products and find the minimum price for each row
brand_products_pricing = all_productos_df[~all_productos_df['id_comercio'].isin([2, 13])].copy()
brand_products_pricing['min_price_calculated'] = (
    brand_products_pricing['precio_unitario_promo2']
    .fillna(brand_products_pricing['precio_unitario_promo1'])
    .fillna(brand_products_pricing['precio_lista'])
)

# 3. Sort by product and calculated price to find the absolute cheapest offering and its store key
cheapest_offerings = (
    brand_products_pricing.sort_values(by=['id_producto', 'min_price_calculated'], ascending=True)
    .groupby('id_producto')
    .first()
    .reset_index()
)

# 4. Join with store details to get the corresponding supermarket brand ('bandera_nombre')
cheapest_offerings_with_brand = pd.merge(
    cheapest_offerings[['id_producto', 'min_price_calculated', 'id_comercio', 'id_bandera', 'id_sucursal']],
    stores_and_brands[['id_comercio', 'id_bandera', 'id_sucursal', 'bandera_nombre']],
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='left'
)

# 5. Build a lookup dataframe for the merge
product_cheapest_lookup = cheapest_offerings_with_brand[['id_producto', 'min_price_calculated', 'bandera_nombre']].rename(columns={
    'min_price_calculated': 'min_price',
    'bandera_nombre': 'cheapest_bandera'
})

# 6. Re-create merged_inferred_and_provided_df with these new metrics merged directly
merged_inferred_and_provided_df = pd.merge(
    grouped_descriptions_df,
    product_details_df[['id_producto', 'marca_ok', 'segmento', 'talle', 'unidades']],
    on='id_producto',
    how='left'
)

# Ensure clean matching keys
merged_inferred_and_provided_df['id_producto'] = merged_inferred_and_provided_df['id_producto'].astype(float)
product_cheapest_lookup['id_producto'] = product_cheapest_lookup['id_producto'].astype(float)

merged_inferred_and_provided_df = pd.merge(
    merged_inferred_and_provided_df,
    product_cheapest_lookup,
    on='id_producto',
    how='left'
)

print("Re-built merged_inferred_and_provided_df with minimum price & supermarket calculated globally:")
merged_inferred_and_provided_df


Re-built merged_inferred_and_provided_df with minimum price & supermarket calculated globally:


,id_producto,all_descriptions,inferred_marca,inferred_segmento,inferred_talle,inferred_unidades,marca_ok,segmento,talle,unidades,min_price,cheapest_bandera
0,7.441008e+12,"HUGGIES DERMACARE PAÑAL P DERMACARE AP, HUGGIE...",Huggies,Supreme Care,P,NaN,NaN,NaN,NaN,NaN,19744.19,Changomas
1,7.500435e+12,PANAL PAMPERS REC.NACIDO RN+ 20 u.,Pampers,Recien Nacido,RN,20.0,NaN,NaN,NaN,NaN,409.60,California Supermercados
2,7.500435e+12,PAÑAL PREMIUM CARE PANTS TALLE: M PAMPERS PAQ ...,Pampers,Premium Care,M,34.0,NaN,NaN,NaN,NaN,31250.00,La Anonima
3,7.500435e+12,PAÑAL PREMIUM CARE PANTS TALLE: G PAMPERS PAQ ...,Pampers,Premium Care,G,30.0,NaN,NaN,NaN,NaN,31250.00,La Anonima
4,7.500435e+12,PAÑAL PREMIUM CARE PANTS TALLE: XG PAMPERS PAQ...,Pampers,Premium Care,XG,26.0,NaN,NaN,NaN,NaN,31250.00,La Anonima
...,...,...,...,...,...,...,...,...,...,...,...,...
178,7.896008e+12,PAÑAL LITTLE SWIMMERS T:P-M HUGGIES PAQ 11 UNI...,Huggies,Little Swimmers,P,11.0,NaN,NaN,NaN,NaN,13869.00,Maxi
179,7.896008e+12,PAÑAL LITTLE SWIMMERS T:G-XG HUGGIES PAQ 10 UN...,Huggies,Little Swimmers,XG,10.0,NaN,NaN,NaN,NaN,9809.00,SuperChangomas
180,7.896062e+12,"BABYSEC PREMIUM PANT PAÑAL PANTS G PREMIUM, BA...",Babysec,Premium,G,NaN,NaN,NaN,NaN,NaN,13759.00,SuperChangomas
181,7.896062e+12,"BABYSEC PREMIUM PANT PAÑAL PANTS XG PREMIUM, B...",Babysec,Premium,XG,NaN,NaN,NaN,NaN,NaN,14319.00,SuperChangomas


In [ ]:
import pandas as pd
import numpy as np

# Ensure product_cheapest_lookup has the necessary columns
# Build the lookup from product_cheapest_lookup variable
product_cheapest_lookup['id_producto'] = product_cheapest_lookup['id_producto'].astype(float)
merged_inferred_and_provided_df['id_producto'] = merged_inferred_and_provided_df['id_producto'].astype(float)

# Safely drop 'min_price' and 'cheapest_bandera' if they already exist to avoid suffix issues
columns_to_drop = [col for col in ['min_price', 'cheapest_bandera'] if col in merged_inferred_and_provided_df.columns]
if columns_to_drop:
    merged_inferred_and_provided_df = merged_inferred_and_provided_df.drop(columns=columns_to_drop)

# Merge again to guarantee 'min_price' presence
merged_inferred_and_provided_df = pd.merge(
    merged_inferred_and_provided_df,
    product_cheapest_lookup[['id_producto', 'min_price', 'cheapest_bandera']],
    on='id_producto',
    how='left'
)

# Coalesce units: prioritize 'unidades' and fall back to 'inferred_unidades'
merged_inferred_and_provided_df['final_units'] = (
    merged_inferred_and_provided_df['unidades']
    .fillna(merged_inferred_and_provided_df['inferred_unidades'])
)

# Calculate the price per diaper
merged_inferred_and_provided_df['price_per_diaper'] = (
    merged_inferred_and_provided_df['min_price'] / merged_inferred_and_provided_df['final_units']
)

# Sort ascending by price per diaper
cheapest_diapers_df = merged_inferred_and_provided_df.sort_values(by='price_per_diaper', ascending=True)

# Save to CSV
output_diaper_csv = 'cheapest_diapers_per_unit.csv'
cheapest_diapers_df.to_csv(output_diaper_csv, float_format=lambda x: f'{np.trunc(x * 100) / 100:.2f}',index=False)

print(f"Saved {len(cheapest_diapers_df)} entries to '{output_diaper_csv}' sorted by price per diaper.")
display(cheapest_diapers_df[['id_producto', 'all_descriptions', 'min_price', 'final_units', 'price_per_diaper', 'cheapest_bandera']].head(15))

# me gustaria subir toda la info yo manual de los ean que corresponden
# me gustaria almacenar
# generaria una columna de seleccion de segmento y talle

Saved 183 entries to 'cheapest_diapers_per_unit.csv' sorted by price per diaper.


,id_producto,all_descriptions,min_price,final_units,price_per_diaper,cheapest_bandera
104,7.794626e+12,PANAL HUGGIES R.NACIDO PREMATURO 30u.,299.0,30.0,9.966667,California Supermercados
108,7.794626e+12,PANAL HUGGIES R.NACIDO PREMATURO 30u.,299.0,30.0,9.966667,California Supermercados
105,7.794626e+12,PANAL HUGGIES R.NACIDO PREMATURO 30u.,299.0,30.0,9.966667,California Supermercados
106,7.794626e+12,PANAL HUGGIES ACTIVE SEC REG.P 12 u.,192.9,12.0,16.075000,California Supermercados
107,7.794626e+12,PANAL HUGGIES ACTIVE SEC REG.G 9 u.,149.9,9.0,16.655556,California Supermercados
1,7.500435e+12,PANAL PAMPERS REC.NACIDO RN+ 20 u.,409.6,20.0,20.480000,California Supermercados
55,7.506195e+12,PANAL PAMPERS RECIEN NACIDO 20 u.,409.6,20.0,20.480000,California Supermercados
56,7.506339e+12,PANAL PAMPERS SUPERSEC NUEVO XXG 8 u.,209.0,8.0,26.125000,California Supermercados
109,7.794626e+12,Toall. Húm. Huggies Triple Protec.x48 PCK-1-un.,5200.0,48.0,108.333333,Vea
71,7.790250e+12,PANALES BABYSEC X8 G U.SOFT PAN B...,998.0,8.0,124.750000,Express


In [ ]:
specific_ean = all_productos_df[all_productos_df['id_producto'] ==  7795323775393]
specific_ean_2 = pd.merge(
    specific_ean,
    all_sucursales_df,
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='left'
)

specific_ean_3 = pd.merge(
    specific_ean_2,
    all_comercio_df,
    on=['id_comercio', 'id_bandera'],
    how='left'
)

# Save to CSV
detailed_product = 'detailed_product.csv'
specific_ean_3.to_csv(detailed_product, index=False)



In [ ]:
all_comercio_df.to_csv('all_comercio_df.csv', index=False)

In [ ]:
import pandas as pd

promo_columns = [
    'precio_unitario_promo1',
    'leyenda_promo1',
    'precio_unitario_promo2',
    'leyenda_promo2'
]

# Identify rows where at least one promo column has a non-null value
has_promo_mask = all_productos_df[promo_columns].notna().any(axis=1)
promos_by_comercio = all_productos_df[has_promo_mask].groupby('id_comercio').size().reset_index(name='rows_with_promos')

print("Commerce IDs ('id_comercio') that contain promotional information:")
display(promos_by_comercio)


Commerce IDs ('id_comercio') that contain promotional information:


,id_comercio,rows_with_promos
0,10,173964
1,12,7900
2,13,63564
3,15,1226357
4,2,53541
5,21,633


Empieza Nutrilon

---



---



---



In [ ]:
print("This runs.")

if True:  # Your error or exit condition
    raise SystemExit("Halting the entire notebook.")

print("This will never run, and subsequent cells are blocked.")


This runs.


SystemExit: Halting the entire notebook.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
nutrilon_productos_df = all_productos_df[all_productos_df['descripcion'].str.contains('nutrilon', case=False, na=False)]


In [ ]:
nutrilon_productos_df

In [ ]:
import re

def infer_state(description):
    desc_lower = str(description).lower()
    if re.search(r'\b(\d+)\s*(ml|cc)\b', desc_lower): # Check for ml or cc first
        return 'liquido'
    if re.search(r'\b(\d+)\s*(gr|g)\b', desc_lower): # Check for gr or g first
        return 'polvo'
    if re.search(r'\b(liquido|fluido)\b', desc_lower):
        return 'liquido'
    if re.search(r'\b(polvo|en polvo)\b', desc_lower):
        return 'polvo'
    return 'otro'

def infer_etapa(description):
    desc_lower = str(description).lower()
    if re.search(r'\b0\s*a\s*6\s*meses\b', desc_lower): # '0 A 6 MESES' for Etapa 1
        return 1
    match = re.search(r'etapa\s*(\d)|et\s*(\d)|\b(\d)\b|pf(\d)', desc_lower) # Handles 'etapa 1', 'et 1', ' 1 ', 'PF1'
    if match:
        # Prioritize 'etapa', then 'et', then standalone digit, then 'pf' digit
        if match.group(1): return int(match.group(1))
        if match.group(2): return int(match.group(2))
        if match.group(3): return int(match.group(3))
        if match.group(4): return int(match.group(4))
    return None

def infer_weight(description):
    desc_lower = str(description).lower()
    # Pattern for ml or cc
    ml_match = re.search(r'(\d+)\s*(ml|cc)\b', desc_lower)
    if ml_match:
        return f"{ml_match.group(1)}{ml_match.group(2)}"

    # Pattern for kg (including 1.2 kg)
    kg_match = re.search(r'(\d+\s*,\s*\d+|\d+\.\d+|\d+)\s*(kg)\b', desc_lower)
    if kg_match:
        # Replace comma with dot for consistent float conversion if necessary
        weight_val = kg_match.group(1).replace(',', '.')
        return f"{float(weight_val)}{kg_match.group(2)}"

    # Pattern for gr (including 800gr explicitly and '800' interpreted as grams)
    gr_match = re.search(r'(\d+)\s*(gr|g)\b', desc_lower)
    if gr_match:
        return f"{gr_match.group(1)}{gr_match.group(2)}"
    # If '800' appears without a unit and no other weight was found, assume '800gr'
    if re.search(r'\b800\b', desc_lower) and not (ml_match or kg_match or gr_match):
        return '800gr'

    return None

def infer_segment(description):
    desc_lower = str(description).lower()
    # Prioritize more specific patterns first
    if re.search(r'\bprofutura\b', desc_lower): # Full 'profutura'
        return 'profutura'
    if re.search(r'\bprofut\b', desc_lower): # 'profut' prefix
        return 'profutura'
    if re.search(r'\b(pro|pf|profu)\b', desc_lower): # 'PRO', 'PF', 'PROFU'
        return 'profutura'
    if re.search(r'\bcomfort\b', desc_lower):
        return 'comfort'
    if re.search(r'\badvance\b', desc_lower):
        return 'advance'
    return 'otros'

# Apply the inference functions to nutrilon_productos_df
nutrilon_productos_inferred_df = nutrilon_productos_df.copy()
nutrilon_productos_inferred_df['estado'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_state)
nutrilon_productos_inferred_df['etapa'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_etapa)
nutrilon_productos_inferred_df['peso'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_weight)
nutrilon_productos_inferred_df['segmento_nutrilon'] = nutrilon_productos_inferred_df['descripcion'].apply(infer_segment)

# Group by id_producto and descripcion, and show the inferred attributes
inferred_summary_df = nutrilon_productos_inferred_df.groupby(['id_producto', 'descripcion']).agg(
    estado=('estado', 'first'),
    etapa=('etapa', 'first'),
    peso=('peso', 'first'),
    segmento_nutrilon=('segmento_nutrilon', 'first')
).reset_index()

print("Nutrilon products with refined inferred attributes:")
display(inferred_summary_df)

In [ ]:
inferred_summary_df

In [ ]:
# 1. Group by id_producto and aggregate all unique descriptions
nutrilon_grouped_descriptions_df = nutrilon_productos_df.groupby('id_producto')['descripcion'].apply(lambda x: ' '.join(x.dropna().unique())).reset_index()
nutrilon_grouped_descriptions_df.rename(columns={'descripcion': 'all_descriptions'}, inplace=True)

# 2. Apply the inference functions to the aggregated descriptions
nutrilon_grouped_descriptions_df['estado'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_state)
nutrilon_grouped_descriptions_df['etapa'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_etapa)
nutrilon_grouped_descriptions_df['peso'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_weight)
nutrilon_grouped_descriptions_df['segmento_nutrilon'] = nutrilon_grouped_descriptions_df['all_descriptions'].apply(infer_segment)

print("Nutrilon products with unique id_producto and inferred attributes from grouped descriptions:")
nutrilon_grouped_descriptions_df

In [ ]:
selected_nutrilons = [
    [7795323000785, "polvo", 2, 0.8],
[7795323000846, "liquido", 2, 0.2],
[7795323002437, "polvo", 2, 0.8],
[7795323002475, "liquido", 2, 0.2],
[7795323772293, "polvo", 2, 0.8],
[7795323772620, "liquido", 2, 0.2],
[7795323775331, "liquido", 2, 0.2],
[7795323775393, "polvo", 2, 1.2]
]

In [ ]:
# 1. Create a DataFrame from selected_nutrilons
nutrilon_headers = ['id_producto', 'estado_target', 'etapa_target', 'peso_target_value']
nutrilon_target_products_df = pd.DataFrame(selected_nutrilons, columns=nutrilon_headers)

# Helper function to determine base unit quantity for price comparison
def get_base_unit_quantity(row):
    estado = str(row['estado_target']).lower()
    peso_value = row['peso_target_value']
    if estado == 'polvo': # Assume kg to grams
        return peso_value * 1000  # Convert kg to grams
    elif estado == 'liquido': # Assume liters to ml
        return peso_value * 1000  # Convert liters to ml
    return None

# Add base_unit_quantity column
nutrilon_target_products_df['base_unit_quantity'] = nutrilon_target_products_df.apply(get_base_unit_quantity, axis=1)

print("Nutrilon Target Products DataFrame:")
display(nutrilon_target_products_df)

In [ ]:
# 2. Filter all_productos_df for these specific Nutrilon product IDs
specific_nutrilon_id_productos = nutrilon_target_products_df['id_producto'].tolist()

filtered_nutrilon_productos_df_temp = all_productos_df[
    all_productos_df['id_producto'].isin(specific_nutrilon_id_productos)
].copy()

# Merge with nutrilon_target_products_df to add the target attributes (estado, etapa, peso)
filtered_nutrilon_products_with_targets = pd.merge(
    filtered_nutrilon_productos_df_temp,
    nutrilon_target_products_df,
    on='id_producto',
    how='left'
)

# Merge with sucursales_2 (closer stores)
merged_nutrilon_with_sucursales = pd.merge(
    filtered_nutrilon_products_with_targets,
    sucursales_2,
    on=['id_comercio', 'id_bandera', 'id_sucursal'],
    how='inner' # Only keep products found in closer stores
)

# Merge with all_comercio_df (commercial entity details)
final_merged_nutrilon_df = pd.merge(
    merged_nutrilon_with_sucursales,
    all_comercio_df,
    on=['id_comercio', 'id_bandera'],
    how='inner'
)

print(f"Found {len(final_merged_nutrilon_df)} entries for selected Nutrilon products merged with store and commercial data.")
display(final_merged_nutrilon_df.head())

In [ ]:
# 3. Select relevant columns and calculate unit price for Nutrilon products

selected_columns_nutrilon = [
    'id_producto',
    'descripcion',
    'marca',
    'precio_lista',
    'precio_referencia',
    'precio_unitario_promo1',
    'leyenda_promo1',
    'precio_unitario_promo2',
    'leyenda_promo2',
    'estado_target',
    'etapa_target',
    'peso_target_value',
    'base_unit_quantity',
    'file_source_x',
    'nombre',
    'calle',
    'numero',
    'bandera_nombre'
]

# Filter the DataFrame to keep only the selected columns
final_merged_nutrilon_selected = final_merged_nutrilon_df[selected_columns_nutrilon].copy()

# Ensure 'base_unit_quantity' is numeric and handle potential NaN values
final_merged_nutrilon_selected['base_unit_quantity'] = pd.to_numeric(final_merged_nutrilon_selected['base_unit_quantity'], errors='coerce')

# Calculate potential unit prices by dividing by 'base_unit_quantity'
promo2_divided_nutrilon = final_merged_nutrilon_selected['precio_unitario_promo2'] / final_merged_nutrilon_selected['base_unit_quantity']
promo1_divided_nutrilon = final_merged_nutrilon_selected['precio_unitario_promo1'] / final_merged_nutrilon_selected['base_unit_quantity']
lista_divided_nutrilon = final_merged_nutrilon_selected['precio_lista'] / final_merged_nutrilon_selected['base_unit_quantity']

# Replace infinite values (e.g., from division by zero) with NaN so fillna can work correctly
promo2_divided_nutrilon.replace([np.inf, -np.inf], np.nan, inplace=True)
promo1_divided_nutrilon.replace([np.inf, -np.inf], np.nan, inplace=True)
lista_divided_nutrilon.replace([np.inf, -np.inf], np.nan, inplace=True)

# Apply coalesce logic: use promo2_divided if available, else promo1_divided, else lista_divided
final_merged_nutrilon_selected['precio_unitario_por_base_unit'] = (
    promo2_divided_nutrilon
    .fillna(promo1_divided_nutrilon)
    .fillna(lista_divided_nutrilon)
)

# Sort the DataFrame by the new calculated unit price to find the cheapest
cheapest_nutrilon_by_ean = final_merged_nutrilon_selected.sort_values(by=['id_producto', 'precio_unitario_por_base_unit'], ascending=True)

print("Cheapest Nutrilon products by EAN (unit price):")
display(cheapest_nutrilon_by_ean[['id_producto', 'descripcion', 'estado_target', 'etapa_target', 'peso_target_value', 'base_unit_quantity', 'precio_unitario_por_base_unit', 'bandera_nombre']].head(10))